# Network Slice DRL — Results Analysis

Run `bash scripts/sync_results.sh` first to pull all CSVs from both GPU servers.

This notebook produces:
- Training curves (acceptance ratio, revenue) per agent × seed
- Final-performance bar charts with 95% CI error bars
- Wilcoxon signed-rank tests (DRL agents vs. best baseline)
- LaTeX-ready summary table

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.2)
RESULTS_DIR = '../results'   # relative to notebooks/
FIGS_DIR    = '../results/figures'
os.makedirs(FIGS_DIR, exist_ok=True)

# Friendly display names
AGENT_LABELS = {
    'dqn_unified':          'DQN (unified)',
    'dqn_separated':        'DQN (separated)',
    'ddqn_unified':         'DDQN (unified)',
    'ddqn_separated':       'DDQN (separated)',
    'greedy_admission':     'Greedy Admission',
    'revenue_heuristic':    'Revenue Heuristic',
    'admission_only_dqn':   'Admission-Only DQN',
}
DRL_AGENTS   = ['dqn_unified', 'dqn_separated', 'ddqn_unified', 'ddqn_separated']
BASELINES    = ['greedy_admission', 'revenue_heuristic', 'admission_only_dqn']
ALL_AGENTS   = DRL_AGENTS + BASELINES

## 1. Load all results

In [ ]:
# --- DRL agent training metrics ---
drl_frames = []
for csv_path in glob.glob(os.path.join(RESULTS_DIR, '**', 'metrics.csv'), recursive=True):
    run_name = os.path.basename(os.path.dirname(csv_path))
    # skip baseline runs (handled separately)
    if run_name.startswith('baselines_'):
        continue
    try:
        df = pd.read_csv(csv_path)
        # infer agent name and seed from run_name, e.g. 'ddqn_unified_s43'
        parts = run_name.rsplit('_s', 1)
        agent = parts[0] if len(parts) == 2 else run_name
        seed  = int(parts[1]) if len(parts) == 2 and parts[1].isdigit() else 0
        df['agent'] = agent
        df['seed']  = seed
        df['run_name'] = run_name
        drl_frames.append(df)
    except Exception as e:
        print(f'Warning: could not load {csv_path}: {e}')

drl_df = pd.concat(drl_frames, ignore_index=True) if drl_frames else pd.DataFrame()
print(f'DRL runs loaded: {len(drl_frames)}  |  rows: {len(drl_df)}')
drl_df.head()

In [ ]:
# --- Baseline evaluation metrics ---
base_frames = []
for csv_path in glob.glob(os.path.join(RESULTS_DIR, 'baselines_s*', 'metrics.csv')):
    try:
        df = pd.read_csv(csv_path)
        base_frames.append(df)
    except Exception as e:
        print(f'Warning: could not load {csv_path}: {e}')

base_df = pd.concat(base_frames, ignore_index=True) if base_frames else pd.DataFrame()
print(f'Baseline rows loaded: {len(base_df)}')
base_df.head()

## 2. Training curves

In [ ]:
if drl_df.empty:
    print('No DRL results yet — run experiments first.')
else:
    METRICS = ['acceptance_ratio', 'fulfillment_ratio', 'revenue', 'mean_ep_reward']
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for ax, metric in zip(axes, METRICS):
        for agent in DRL_AGENTS:
            sub = drl_df[drl_df['agent'] == agent]
            if sub.empty:
                continue
            grouped = sub.groupby('episode')[metric]
            mean = grouped.mean()
            std  = grouped.std().fillna(0)
            ax.plot(mean.index, mean.values, label=AGENT_LABELS.get(agent, agent))
            ax.fill_between(mean.index,
                            mean.values - std.values,
                            mean.values + std.values, alpha=0.15)
        ax.set_xlabel('Episode')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(metric.replace('_', ' ').title())
        ax.legend(fontsize=9)

    fig.suptitle('Training Curves (mean ± std over 5 seeds)', fontsize=13)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, 'training_curves.pdf'), bbox_inches='tight')
    plt.show()
    print(f'Saved → {FIGS_DIR}/training_curves.pdf')

## 3. Final-episode performance (last 10% of training)

In [ ]:
def _final_window(df: pd.DataFrame, frac: float = 0.1) -> pd.DataFrame:
    """Return rows from the last *frac* fraction of episodes per run."""
    out = []
    for run, g in df.groupby('run_name'):
        cutoff = g['episode'].max() * (1 - frac)
        out.append(g[g['episode'] >= cutoff])
    return pd.concat(out, ignore_index=True) if out else df


if not drl_df.empty:
    final_drl = (
        _final_window(drl_df)
        .groupby(['agent', 'seed'])[['acceptance_ratio', 'fulfillment_ratio', 'revenue']]
        .mean()
        .reset_index()
    )
    print('DRL final-window means per seed:')
    display(final_drl.groupby('agent').mean(numeric_only=True).round(4))

In [ ]:
# Combine DRL final metrics with baseline eval metrics for a unified comparison
rows = []

if not drl_df.empty:
    for _, r in final_drl.iterrows():
        rows.append({'agent': r['agent'], 'seed': r['seed'],
                     'acceptance_ratio': r['acceptance_ratio'],
                     'fulfillment_ratio': r['fulfillment_ratio'],
                     'revenue': r['revenue']})

if not base_df.empty:
    # Use unified mode only for fair comparison
    for _, r in base_df[base_df.get('mode', 'unified') == 'unified'].iterrows():
        rows.append({'agent': r['baseline'], 'seed': r.get('seed', 0),
                     'acceptance_ratio': r.get('acceptance_ratio', np.nan),
                     'fulfillment_ratio': r.get('fulfillment_ratio', np.nan),
                     'revenue': r.get('revenue', np.nan)})

compare_df = pd.DataFrame(rows)
print(compare_df.groupby('agent').agg(['mean', 'std']).round(4))

## 4. Comparison bar chart

In [ ]:
if not compare_df.empty:
    METRICS_BAR = ['acceptance_ratio', 'fulfillment_ratio', 'revenue']
    agg = compare_df.groupby('agent')[METRICS_BAR].agg(['mean', 'std']).reset_index()
    agg.columns = ['agent'] + [f'{m}_{s}' for m in METRICS_BAR for s in ('mean', 'std')]

    order = [a for a in ALL_AGENTS if a in agg['agent'].values]
    agg = agg.set_index('agent').loc[order].reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, metric in zip(axes, METRICS_BAR):
        colors = ['steelblue' if a in DRL_AGENTS else 'coral' for a in agg['agent']]
        ax.bar(range(len(agg)),
               agg[f'{metric}_mean'],
               yerr=agg[f'{metric}_std'],
               color=colors, capsize=4, error_kw={'elinewidth': 1.2})
        ax.set_xticks(range(len(agg)))
        ax.set_xticklabels([AGENT_LABELS.get(a, a) for a in agg['agent']],
                           rotation=35, ha='right', fontsize=9)
        ax.set_title(metric.replace('_', ' ').title())
        ax.set_ylabel('Mean ± Std')

    from matplotlib.patches import Patch
    handles = [Patch(color='steelblue', label='DRL agents'),
               Patch(color='coral',     label='Baselines')]
    fig.legend(handles=handles, loc='lower right', ncol=2, fontsize=9)
    fig.suptitle('Final Performance Comparison (5 seeds)', fontsize=13)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, 'comparison_bar.pdf'), bbox_inches='tight')
    plt.show()
    print(f'Saved → {FIGS_DIR}/comparison_bar.pdf')

## 5. Statistical significance (Wilcoxon signed-rank)

In [ ]:
if not compare_df.empty:
    METRIC = 'acceptance_ratio'
    best_baseline = 'revenue_heuristic'   # update after seeing results
    sig_rows = []

    bl_vals = compare_df[compare_df['agent'] == best_baseline][METRIC].dropna().values

    for agent in DRL_AGENTS:
        agent_vals = compare_df[compare_df['agent'] == agent][METRIC].dropna().values
        if len(agent_vals) < 2 or len(bl_vals) < 2:
            continue
        n = min(len(agent_vals), len(bl_vals))
        stat, pval = stats.wilcoxon(agent_vals[:n], bl_vals[:n], alternative='greater')
        sig_rows.append({
            'agent': AGENT_LABELS.get(agent, agent),
            'baseline': best_baseline,
            'metric': METRIC,
            'statistic': round(stat, 3),
            'p_value': round(pval, 4),
            'significant (p<0.05)': pval < 0.05,
        })

    sig_df = pd.DataFrame(sig_rows)
    display(sig_df)

## 6. LaTeX summary table

In [ ]:
if not compare_df.empty:
    tex_rows = []
    for agent in order:
        sub = compare_df[compare_df['agent'] == agent]
        for metric in ['acceptance_ratio', 'fulfillment_ratio', 'revenue']:
            m = sub[metric].mean()
            s = sub[metric].std()
            tex_rows.append({'Agent': AGENT_LABELS.get(agent, agent),
                             'Metric': metric, 'Mean': m, 'Std': s})

    tex_df = pd.DataFrame(tex_rows).pivot_table(
        index='Agent', columns='Metric', values=['Mean', 'Std'], aggfunc='first'
    )

    latex_path = os.path.join(FIGS_DIR, 'results_table.tex')
    tex_df.round(4).to_latex(latex_path, escape=False)
    print(f'LaTeX table saved → {latex_path}')
    print(tex_df.round(4).to_string())